# RelayOps — Fine-tune the Tier-1 intent classifier (Colab)

Fine-tunes **Qwen2.5-1.5B-Instruct** with **Unsloth + LoRA** on the RelayOps intent dataset, then evaluates it against the keyword and Complement-NB baselines on the same held-out + adversarial sets.

**Before running:**
1. `Runtime -> Change runtime type -> Hardware accelerator: GPU` (a free **T4** is enough).
2. Edit `REPO_URL` in step 2 to point at your RelayOps repo.
3. `Runtime -> Run all`.

**What to send back to Claude:** the printed output of step 5 (the eval table + confusion matrices) and either the Hugging Face repo id from step 6 or the downloaded adapter from step 7.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

## 1. Install dependencies
If Unsloth errors on install, use the current official Colab snippet from https://github.com/unslothai/unsloth (the API moves fast).

In [ ]:
%pip install -q unsloth
%pip install -q trl peft datasets transformers huggingface_hub

## 2. Clone your RelayOps repo

In [ ]:
REPO_URL = "https://github.com/<your-username>/relayops.git"  # <-- EDIT THIS

import os, shutil
if os.path.isdir("relayops"):
    shutil.rmtree("relayops")
!git clone $REPO_URL relayops
%cd relayops

## 3. Export the fine-tuning dataset (chat JSONL, 70/15/15)

In [ ]:
!python -m src.eval.export_finetune_data
print("\nsample line:")
!head -1 src/eval/data/finetune/train.jsonl

## 4. Train the LoRA adapter
Runs the recipe in `src/router/finetune_train.py` (Unsloth + QLoRA, 3 epochs). Saves the adapter to `models/intent-qwen2.5-1.5b-lora/`. Takes only a few minutes on a T4 for this small dataset.

In [ ]:
!python -m src.router.finetune_train

## 5. Evaluate: fine-tuned vs keyword vs Complement NB
Setting `RELAYOPS_INTENT_MODEL` makes the eval runner load and score the fine-tuned model on the held-out test set and the adversarial set, alongside the baselines.

**Copy this whole output and send it back.**

In [ ]:
!RELAYOPS_INTENT_MODEL=models/intent-qwen2.5-1.5b-lora python -m src.eval.run_intent_eval

## 6. (Recommended) Push the adapter to the Hugging Face Hub
Cleanest way to get the model back into the repo — load it later with `RELAYOPS_INTENT_MODEL=<your-username>/relayops-intent-qwen`. You'll need a write token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login, HfApi

HF_REPO = "<your-username>/relayops-intent-qwen"  # <-- EDIT THIS

login()  # paste a token with write access
api = HfApi()
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path="models/intent-qwen2.5-1.5b-lora", repo_id=HF_REPO, repo_type="model")
print("pushed ->", HF_REPO)

## 7. (Alternative) Download the adapter as a zip
Use this instead of step 6 if you'd rather hand over the file directly.

In [ ]:
import shutil
shutil.make_archive("intent-lora", "zip", "models/intent-qwen2.5-1.5b-lora")
from google.colab import files
files.download("intent-lora.zip")

## What to send back
1. The **full output of step 5** (accuracy + confusion matrices) — I'll update the README table and docs with the real fine-tuned numbers.
2. Either the **HF repo id** from step 6, or the **`intent-lora.zip`** from step 7.

Load it anywhere with: `RELAYOPS_INTENT_MODEL=<hf-repo-id-or-local-dir> python -m src.eval.run_intent_eval`, or in code via `router.registry.get_classifier("finetuned")`.